# [12.1] CLIP, SigLIP, and VLM Controls - Exercises

Build the local control ladder for vision-language interpretability claims: contrastive retrieval, SigLIP loss, visual-token localization, synthetic scene controls, image-vs-text baselines, object-region patching, hallucination checks, and modality arbitration.

In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t
import torch.nn.functional as F

chapter = "chapter12_vlm_interpretability"
section = "part1_clip_siglip_vlm_controls"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_clip_siglip_vlm_controls.tests as tests

GT_TIER = "GT-1"
EXERCISE_ID = "12_1_clip_siglip_vlm_controls"
EXPECTED_RUNTIME = "45-70 minutes for exercises; several minutes for cached CUDA report regeneration"
REQUIRES_GPU = True


In [ ]:
@dataclass(frozen=True)
class ContrastiveAlignmentReport:
    image_to_text_accuracy: float
    text_to_image_accuracy: float
    mean_positive_margin: float
    aligned: bool


@dataclass(frozen=True)
class VisualTokenAttributionReport:
    token_scores: t.Tensor
    top_token_indices: t.Tensor
    top_token_mass: float
    localized: bool


@dataclass(frozen=True)
class ObjectHallucinationReport:
    object_score: float
    visual_evidence_score: float
    text_prior_score: float
    text_prior_gap: float
    flags_hallucination: bool


@dataclass(frozen=True)
class ModalityArbitrationReport:
    visual_choice_score: float
    text_prior_choice_score: float
    visual_margin: float
    trusts_visual_evidence: bool


@dataclass(frozen=True)
class SyntheticVLMScene:
    image_id: str
    shape: str
    color: str
    bbox: tuple[float, float, float, float]
    question: str
    answer: str
    counterfactual_answer: str
    spurious_text: str | None
    split: str


@dataclass(frozen=True)
class SyntheticClothingScene:
    image_id: str
    garment_type: str
    color: str
    style: str
    bbox: tuple[float, float, float, float]
    question: str
    answer: str
    counterfactual_answer: str
    spurious_text: str | None
    split: str


@dataclass(frozen=True)
class ClothingGeometryReport:
    garment_accuracy: float
    color_accuracy: float
    style_accuracy: float
    text_prior_color_agreement: float
    random_color_agreement: float
    predicts_clothing_factors: bool
    rejects_text_prior: bool
    rejects_random_labels: bool


@dataclass(frozen=True)
class ControlledVLMBaselineReport:
    joint_accuracy: float
    image_only_accuracy: float
    text_only_accuracy: float
    joint_beats_text_only: bool
    text_only_fails_image_questions: bool


@dataclass(frozen=True)
class VisualRegionPatchReport:
    clean_margin: float
    object_patch_margin: float
    background_patch_margin: float
    random_patch_margin: float
    object_patch_effect: float
    background_patch_effect: float
    random_patch_effect: float
    object_beats_background: bool
    object_beats_random: bool
    flips_answer: bool


@dataclass(frozen=True)
class VisualSequencePatchReport:
    clean_margins: list[float]
    corrupt_margins: list[float]
    object_patch_margins: list[float]
    background_patch_margins: list[float]
    random_patch_margins: list[float]
    full_sequence_patch_margins: list[float]
    object_patch_effects: list[float]
    background_patch_effects: list[float]
    random_patch_effects: list[float]
    full_sequence_patch_effects: list[float]
    min_object_gap_over_background: float
    min_object_gap_over_random: float
    full_sequence_patch_max_abs_margin_error: float
    object_patch_flips_answer: bool
    background_patch_preserves_answer: bool
    random_patch_preserves_answer: bool
    full_sequence_patch_flips_answer: bool
    full_sequence_patch_matches_corrupt: bool
    object_beats_background: bool
    object_beats_random: bool
    passes_activation_patching_controls: bool


In [ ]:
def _l2_normalize(values: t.Tensor, *, eps: float = 1e-8) -> t.Tensor:
    return values.float() / values.float().norm(dim=-1, keepdim=True).clamp_min(eps)


def clip_contrastive_logits(
    image_embeddings: t.Tensor,
    text_embeddings: t.Tensor,
    *,
    logit_scale: float = 10.0,
) -> t.Tensor:
    raise NotImplementedError()


def contrastive_alignment_report(
    logits: t.Tensor,
    *,
    min_accuracy: float = 1.0,
    min_positive_margin: float = 1.0,
) -> ContrastiveAlignmentReport:
    raise NotImplementedError()


def contrastive_smoke_test() -> dict:
    image_embeddings = t.eye(3)
    text_embeddings = t.eye(3)
    logits = clip_contrastive_logits(image_embeddings, text_embeddings, logit_scale=5.0)
    return contrastive_alignment_report(logits, min_accuracy=1.0, min_positive_margin=4.0).__dict__


tests.test_contrastive_smoke_test(contrastive_smoke_test)

In [ ]:
def siglip_pairwise_loss(logits: t.Tensor, labels: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


def siglip_smoke_test() -> dict:
    logits = t.tensor([[4.0, -4.0], [-3.0, 3.0]])
    labels = t.eye(2)
    return {"loss": siglip_pairwise_loss(logits, labels).item()}


tests.test_siglip_smoke_test(siglip_smoke_test)

In [ ]:
def visual_token_attribution_report(
    token_activations: t.Tensor,
    text_direction: t.Tensor,
    *,
    top_k: int = 2,
    min_top_token_mass: float = 0.6,
) -> VisualTokenAttributionReport:
    raise NotImplementedError()


def token_attribution_smoke_test() -> dict:
    token_activations = t.tensor([[0.0, 0.0], [3.0, 0.0], [2.0, 0.0], [0.0, 1.0]])
    report = visual_token_attribution_report(token_activations, t.tensor([1.0, 0.0]), top_k=2, min_top_token_mass=0.8)
    result = report.__dict__.copy()
    result["token_scores"] = report.token_scores.tolist()
    result["top_token_indices"] = report.top_token_indices.tolist()
    return result


tests.test_token_attribution_smoke_test(token_attribution_smoke_test)

In [ ]:
def generate_synthetic_colored_shape_scenes(
    *,
    colors: tuple[str, ...] = ("red", "blue"),
    shapes: tuple[str, ...] = ("cube", "sphere"),
    split: str = "train",
    include_spurious_text: bool = True,
) -> tuple[SyntheticVLMScene, ...]:
    raise NotImplementedError()


def generate_synthetic_clothing_scenes(
    *,
    garment_types: tuple[str, ...] = ("shirt", "coat"),
    colors: tuple[str, ...] = ("red", "blue"),
    styles: tuple[str, ...] = ("formal", "athletic"),
    split: str = "train",
    include_spurious_text: bool = True,
) -> tuple[SyntheticClothingScene, ...]:
    raise NotImplementedError()


def synthetic_scene_schema_smoke_test() -> dict:
    scenes = generate_synthetic_colored_shape_scenes(colors=("red", "blue"), shapes=("cube", "sphere"), split="train")
    return {
        "num_scenes": len(scenes),
        "first_scene": scenes[0].__dict__,
        "has_spurious_text_control": all(scene.spurious_text is not None for scene in scenes),
        "has_counterfactual_answers": all(scene.answer != scene.counterfactual_answer for scene in scenes),
    }


tests.test_synthetic_scene_schema_smoke_test(synthetic_scene_schema_smoke_test)

In [ ]:
def _nearest_centroid_predictions(train_points: t.Tensor, train_labels: t.Tensor, heldout_points: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


def _accuracy(predictions: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


def clothing_geometry_report(
    train_embeddings: t.Tensor,
    heldout_embeddings: t.Tensor,
    train_garment_labels: t.Tensor,
    heldout_garment_labels: t.Tensor,
    train_color_labels: t.Tensor,
    heldout_color_labels: t.Tensor,
    train_style_labels: t.Tensor,
    heldout_style_labels: t.Tensor,
    text_prior_color_labels: t.Tensor,
    random_color_labels: t.Tensor,
    *,
    min_factor_accuracy: float = 0.9,
    max_text_prior_agreement: float = 0.5,
    max_random_agreement: float = 0.5,
) -> ClothingGeometryReport:
    raise NotImplementedError()


# After implementing the three functions above and `generate_synthetic_clothing_scenes`,
# call `tests.test_clothing_geometry_smoke_test(clothing_geometry_smoke_test)` using
# the wrapper from the instruction page.

In [ ]:
def controlled_vlm_baseline_report(
    joint_logits: t.Tensor,
    image_only_logits: t.Tensor,
    text_only_logits: t.Tensor,
    labels: t.Tensor,
    *,
    min_joint_accuracy: float = 0.9,
    min_image_only_accuracy: float = 0.9,
    max_text_only_accuracy: float = 0.5,
) -> ControlledVLMBaselineReport:
    raise NotImplementedError()


def visual_region_patch_report(
    clean_token_contributions: t.Tensor,
    corrupt_token_contributions: t.Tensor,
    *,
    object_token_indices: tuple[int, ...] | list[int],
    background_token_indices: tuple[int, ...] | list[int],
    target_index: int,
    counterfactual_index: int,
    random_token_indices: tuple[int, ...] | list[int] | None = None,
    min_object_gap: float = 0.5,
) -> VisualRegionPatchReport:
    raise NotImplementedError()



def visual_sequence_patch_report(
    clean_logits: t.Tensor,
    corrupt_logits: t.Tensor,
    object_patch_logits: t.Tensor,
    background_patch_logits: t.Tensor,
    random_patch_logits: t.Tensor,
    full_sequence_patch_logits: t.Tensor,
    *,
    target_indices: t.Tensor,
    counterfactual_indices: t.Tensor,
    min_object_gap: float = 1.0,
    max_full_sequence_margin_error: float = 1e-3,
) -> VisualSequencePatchReport:
    raise NotImplementedError()


def visual_sequence_patch_smoke_test(device: str | t.device = "cpu") -> dict:
    device = t.device(device)
    clean_logits = t.tensor([[4.0, 0.0], [0.0, 4.0]], device=device)
    corrupt_logits = t.tensor([[0.0, 4.0], [4.0, 0.0]], device=device)
    background_logits = t.tensor([[3.8, 0.0], [0.0, 3.8]], device=device)
    random_logits = t.tensor([[3.7, 0.0], [0.0, 3.7]], device=device)
    return visual_sequence_patch_report(
        clean_logits,
        corrupt_logits,
        corrupt_logits,
        background_logits,
        random_logits,
        corrupt_logits,
        target_indices=t.tensor([0, 1], device=device),
        counterfactual_indices=t.tensor([1, 0], device=device),
        min_object_gap=1.0,
    ).__dict__.copy()


tests.test_visual_sequence_patch_smoke_test(visual_sequence_patch_smoke_test)


def object_hallucination_report(
    *,
    object_score: float,
    visual_evidence_score: float,
    text_prior_score: float,
    min_object_score: float = 0.7,
    min_visual_evidence: float = 0.5,
    min_text_prior_gap: float = 0.2,
) -> ObjectHallucinationReport:
    raise NotImplementedError()


def modality_arbitration_report(
    candidate_scores: t.Tensor,
    *,
    visual_index: int,
    text_prior_index: int,
    min_visual_margin: float = 0.1,
) -> ModalityArbitrationReport:
    raise NotImplementedError()


def hallucination_smoke_test() -> dict:
    return object_hallucination_report(
        object_score=0.9,
        visual_evidence_score=0.2,
        text_prior_score=0.8,
        min_object_score=0.7,
        min_visual_evidence=0.5,
        min_text_prior_gap=0.4,
    ).__dict__


def arbitration_smoke_test() -> dict:
    candidate_scores = t.tensor([0.1, 0.8, 0.2])
    return modality_arbitration_report(
        candidate_scores,
        visual_index=1,
        text_prior_index=2,
        min_visual_margin=0.5,
    ).__dict__


tests.test_hallucination_smoke_test(hallucination_smoke_test)
tests.test_arbitration_smoke_test(arbitration_smoke_test)


def clothing_geometry_smoke_test(device: str | t.device = "cpu") -> dict:
    device = t.device(device)
    scenes = generate_synthetic_clothing_scenes(split="heldout")
    train_embeddings = t.tensor(
        [
            [3.0, 0.0, 2.0, 0.0, 1.5, 0.0],
            [3.0, 0.0, 0.0, 2.0, 0.0, 1.5],
            [0.0, 3.0, 2.0, 0.0, 0.0, 1.5],
            [0.0, 3.0, 0.0, 2.0, 1.5, 0.0],
        ],
        device=device,
    )
    heldout_embeddings = train_embeddings + 0.05
    garment_labels = t.tensor([0, 0, 1, 1], device=device)
    color_labels = t.tensor([0, 1, 0, 1], device=device)
    style_labels = t.tensor([0, 1, 1, 0], device=device)
    text_prior_color_labels = 1 - color_labels
    random_color_labels = t.tensor([0, 0, 1, 1], device=device)
    report = clothing_geometry_report(
        train_embeddings,
        heldout_embeddings,
        garment_labels,
        garment_labels,
        color_labels,
        color_labels,
        style_labels,
        style_labels,
        text_prior_color_labels,
        random_color_labels,
    )
    result = report.__dict__.copy()
    result["scene_count"] = len(scenes)
    result["first_scene"] = scenes[0].__dict__
    result["has_spurious_text_control"] = all(scene.spurious_text for scene in scenes)
    result["text_prior_color_labels"] = text_prior_color_labels.detach().cpu().tolist()
    result["random_color_labels"] = random_color_labels.detach().cpu().tolist()
    result["random_labels_distinct_from_text_prior"] = not t.equal(random_color_labels, text_prior_color_labels)
    result["random_labels_distinct_from_true_labels"] = not t.equal(random_color_labels, color_labels)
    result["random_label_seed"] = 0
    return result


def controlled_baselines_smoke_test() -> dict:
    labels = t.tensor([0, 1, 0, 1])
    joint_logits = t.tensor([[3.0, 0.0], [0.0, 3.0], [2.0, 0.0], [0.0, 2.0]])
    image_only_logits = t.tensor([[2.0, 0.0], [0.0, 2.0], [1.5, 0.0], [0.0, 1.5]])
    text_only_logits = t.tensor([[0.0, 1.0], [0.0, 1.0], [0.0, 1.0], [0.0, 1.0]])
    return controlled_vlm_baseline_report(
        joint_logits,
        image_only_logits,
        text_only_logits,
        labels,
        min_joint_accuracy=1.0,
        min_image_only_accuracy=1.0,
        max_text_only_accuracy=0.5,
    ).__dict__


def visual_region_patch_smoke_test(device: str | t.device = "cpu") -> dict:
    device = t.device(device)
    clean_contributions = t.tensor(
        [[2.0, -1.0], [1.0, -0.5], [0.1, 0.0], [0.0, 0.1]],
        device=device,
    )
    corrupt_contributions = t.tensor(
        [[-1.0, 2.0], [-0.5, 1.0], [0.1, 0.0], [0.0, 0.1]],
        device=device,
    )
    object_token_indices = [0, 1]
    background_token_indices = [2]
    random_token_indices = [2, 3]
    result = visual_region_patch_report(
        clean_contributions,
        corrupt_contributions,
        object_token_indices=object_token_indices,
        background_token_indices=background_token_indices,
        random_token_indices=random_token_indices,
        target_index=0,
        counterfactual_index=1,
        min_object_gap=1.0,
    ).__dict__
    result["object_token_count"] = len(object_token_indices)
    result["background_token_count"] = len(background_token_indices)
    result["random_token_count"] = len(random_token_indices)
    result["random_control_same_size"] = len(random_token_indices) == len(object_token_indices)
    return result


tests.test_clothing_geometry_smoke_test(clothing_geometry_smoke_test)
tests.test_controlled_baselines_smoke_test(controlled_baselines_smoke_test)
tests.test_visual_region_patch_smoke_test(visual_region_patch_smoke_test)


def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "contrastive": contrastive_smoke_test(),
        "siglip": siglip_smoke_test(),
        "token_attribution": token_attribution_smoke_test(),
        "hallucination": hallucination_smoke_test(),
        "arbitration": arbitration_smoke_test(),
        "synthetic_scene_schema": synthetic_scene_schema_smoke_test(),
        "clothing_geometry": clothing_geometry_smoke_test(),
        "controlled_baselines": controlled_baselines_smoke_test(),
        "visual_region_patch": visual_region_patch_smoke_test(),
        "visual_sequence_patch": visual_sequence_patch_smoke_test(),
    }


tests.test_notebook_contract(run_smoke_test)


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    tests.test_committed_verification_report_real_model_controls(report)
    gpu = report["metrics"]["gpu_test"]
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = run_gpu_test(max_vram_gb=24.0)
{key: gpu[key] for key in [
    "real_clip_rendered_shape_preflight_passed",
    "real_clip_visual_token_activation_patching_preflight_passed",
    "real_siglip_rendered_shape_preflight_passed",
    "real_siglip_visual_token_activation_patching_preflight_passed",
    "real_qwen25_vl_generation_preflight_passed",
    "object_beats_background",
    "object_beats_random",
    "random_patch_same_size",
    "clothing_random_color_agreement",
    "peak_vram_gb",
]}
